# EXECUÇÃO DOS MODELOS COMPARATIVOS - TODOS OS EXPERIMENTOS

Este notebook executa todos os modelos comparativos (HAT, ARF, SRP, ACDWM, ERulesD2S, ROSE) em **todos os 3 experimentos**:

| Experimento | chunk_size | PENALTY_WEIGHT | Diretórios |
|-------------|------------|----------------|------------|
| **EXP-A** | 1000 | 0.0 | experiments_6chunks_phase2_gbml, experiments_6chunks_phase3_real |
| **EXP-B** | 2000 | 0.0 | experiments_chunk2000_phase1, experiments_chunk2000_phase2 |
| **EXP-C** | 2000 | 0.1 | experiments_balanced_phase1, experiments_balanced_phase2 |

**Modelos:**
- ROSE_Original (prequential, como no paper)
- ROSE_ChunkEval (avaliação por chunk, comparável com GBML)
- HAT (Hoeffding Adaptive Tree)
- ARF (Adaptive Random Forest)
- SRP (Streaming Random Patches)
- ACDWM (apenas binário)
- ERulesD2S (cache existente)

**Importante:** Os resultados são salvos nas mesmas pastas do GBML para facilitar a comparação.

---
## PARTE 1: SETUP DO AMBIENTE
---

In [ ]:
# CÉLULA 1.1: Montar Google Drive
from google.colab import drive
import os
import sys
from pathlib import Path

# Montar Drive
drive.mount('/content/drive')

# Diretório de trabalho no Drive
DRIVE_PATH = '/content/drive/MyDrive/DSL-AG-hybrid'
WORK_DIR = '/content/DSL-AG-hybrid'

# Copiar para /content (mais rápido)
if not os.path.exists(WORK_DIR):
    print(f"Copiando {DRIVE_PATH} para {WORK_DIR}...")
    !cp -r "{DRIVE_PATH}" "{WORK_DIR}"
    print("Cópia concluída!")
else:
    print(f"Diretório {WORK_DIR} já existe.")

os.chdir(WORK_DIR)
print(f"Diretório de trabalho: {os.getcwd()}")

In [ ]:
# CÉLULA 1.2: Instalar Java 11 e Maven (necessário para ROSE e ERulesD2S)
#
# Java 11 é requerido pelo ERulesD2S (JCLEC4)
# Maven é necessário para compilar dependências Java

print("Instalando Java 11 e Maven...")

!apt-get update -qq
!apt-get install -y -qq openjdk-11-jdk maven > /dev/null 2>&1

# Configurar JAVA_HOME (importante para ERulesD2S)
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

# Verificar instalações
print("\n--- Java ---")
!java -version

print("\n--- Maven ---")
!mvn -version 2>/dev/null | head -3

print("\nJava 11 e Maven instalados com sucesso!")
print(f"JAVA_HOME: {os.environ.get('JAVA_HOME', 'NÃO DEFINIDO')}")

In [ ]:
# CÉLULA 1.3: Instalar Dependências Python
print("Instalando dependências Python...")
!pip install -q river scikit-learn pandas numpy scipy scikit-posthocs matplotlib seaborn
print("Dependências instaladas!")

In [ ]:
# CÉLULA 1.4: Baixar JARs do ROSE
import urllib.request
from pathlib import Path

rose_dir = Path('rose_jars')
rose_dir.mkdir(exist_ok=True)

# URLs dos JARs
ROSE_JARS = {
    'ROSE-1.0.jar': 'https://github.com/canoalberto/ROSE/releases/download/v1.0/ROSE-1.0.jar',
    'MOA-dependencies.jar': 'https://github.com/canoalberto/ROSE/releases/download/v1.0/MOA-dependencies.jar',
    'sizeofag-1.0.4.jar': 'https://repo1.maven.org/maven2/com/github/fracpete/sizeofag/1.0.4/sizeofag-1.0.4.jar'
}

for jar_name, url in ROSE_JARS.items():
    jar_path = rose_dir / jar_name
    if not jar_path.exists():
        print(f"Baixando {jar_name}...")
        urllib.request.urlretrieve(url, jar_path)
    print(f"  {jar_name}: OK")

print("\nROSE configurado!")

In [ ]:
# CÉLULA 1.5: Clonar ACDWM Repository
ACDWM_DIR = "/content/ACDWM"

if not os.path.exists(ACDWM_DIR):
    print("Clonando repositório ACDWM...")
    !git clone https://github.com/jasonyanglu/ACDWM.git {ACDWM_DIR}
else:
    print("ACDWM já clonado.")

# Verificar arquivos
dwmil_file = Path(ACDWM_DIR) / "dwmil.py"
print(f"dwmil.py: {'OK' if dwmil_file.exists() else 'FALTANDO'}")

In [ ]:
# CÉLULA 1.6: Setup ERulesD2S
#
# ERulesD2S requer:
# - Java 11 (instalado na célula anterior)
# - Maven (instalado na célula anterior)
# - erulesd2s.jar
# - JCLEC4-base-1.0-jar-with-dependencies.jar (na pasta lib/)

print("="*60)
print("CONFIGURANDO ERulesD2S")
print("="*60)

# Verificar JAVA_HOME
java_home = os.environ.get('JAVA_HOME', '')
if java_home:
    print(f"JAVA_HOME: {java_home}")
else:
    print("AVISO: JAVA_HOME não definido - definindo agora...")
    os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
    print(f"JAVA_HOME: {os.environ['JAVA_HOME']}")

# Verificar se JARs existem
erulesd2s_jar = Path(WORK_DIR) / "erulesd2s.jar"
jclec_jar = Path(WORK_DIR) / "lib" / "JCLEC4-base-1.0-jar-with-dependencies.jar"

print(f"\nVerificando JARs...")
print(f"  erulesd2s.jar: {'ENCONTRADO' if erulesd2s_jar.exists() else 'NÃO ENCONTRADO'}")
print(f"  JCLEC4 JAR: {'ENCONTRADO' if jclec_jar.exists() else 'NÃO ENCONTRADO'}")

# Se JAR não existe, executar setup
if not erulesd2s_jar.exists():
    print("\nExecutando setup_erulesd2s.py...")
    setup_script = Path(WORK_DIR) / "setup_erulesd2s.py"
    
    if setup_script.exists():
        # Executar setup (pode demorar alguns minutos)
        !cd {WORK_DIR} && python setup_erulesd2s.py
        
        # Verificar novamente
        if erulesd2s_jar.exists():
            print("\nERulesD2S configurado com sucesso!")
            print(f"  JAR: {erulesd2s_jar}")
            
            # Verificar JCLEC4
            if jclec_jar.exists():
                print(f"  JCLEC4: {jclec_jar}")
            else:
                print("  AVISO: JCLEC4 JAR não encontrado - ERulesD2S pode falhar!")
        else:
            print("\nERRO: erulesd2s.jar não foi criado!")
            print("ERulesD2S será ignorado nos experimentos.")
    else:
        print(f"\nERRO: {setup_script} não encontrado!")
        print("ERulesD2S será ignorado nos experimentos.")
else:
    print("\nERulesD2S já está configurado!")
    
    # Verificar tamanho do JAR (deve ser grande, ~58MB)
    jar_size = erulesd2s_jar.stat().st_size / (1024 * 1024)
    print(f"  Tamanho: {jar_size:.1f} MB")
    
    # Verificar JCLEC4 mesmo assim
    if not jclec_jar.exists():
        print("  AVISO: JCLEC4 JAR não encontrado - ERulesD2S pode falhar!")
    else:
        jclec_size = jclec_jar.stat().st_size / (1024 * 1024)
        print(f"  JCLEC4: {jclec_size:.1f} MB")

# Listar pasta lib se existir
lib_dir = Path(WORK_DIR) / "lib"
if lib_dir.exists():
    print(f"\nConteúdo da pasta lib/:")
    !ls -lh {lib_dir}/*.jar 2>/dev/null | head -5

# Teste rápido: verificar se Java consegue carregar o JAR
if erulesd2s_jar.exists():
    print("\nTeste rápido do JAR...")
    test_result = !java -cp {erulesd2s_jar} moa.DoTask 2>&1 | head -3
    if test_result:
        print("  JAR carregável: OK")
    else:
        print("  AVISO: Não foi possível verificar o JAR")

print("\n" + "="*60)

In [ ]:
# CÉLULA 1.6: Verificação do Ambiente
print("="*80)
print("VERIFICAÇÃO DO AMBIENTE")
print("="*80)

# 1. Java
print("\n1. Java")
!java -version 2>&1 | head -1

# 2. ROSE JARs
print("\n2. ROSE JARs")
for jar in ['ROSE-1.0.jar', 'MOA-dependencies.jar', 'sizeofag-1.0.4.jar']:
    exists = (Path('rose_jars') / jar).exists()
    print(f"   {jar}: {'OK' if exists else 'FALTANDO'}")

# 3. ACDWM
print("\n3. ACDWM")
print(f"   dwmil.py: {'OK' if Path(ACDWM_DIR + '/dwmil.py').exists() else 'FALTANDO'}")

# 4. River
print("\n4. River")
try:
    import river
    from river import tree, ensemble, forest
    print(f"   River version: {river.__version__}")
    print(f"   tree.HoeffdingAdaptiveTreeClassifier: OK")
    print(f"   forest.ARFClassifier: OK")
    print(f"   ensemble.SRPClassifier: OK")
except Exception as e:
    print(f"   River: ERRO - {e}")

# 5. ERulesD2S
print("\n5. ERulesD2S")
erulesd2s_jar = Path(WORK_DIR) / "erulesd2s.jar"
print(f"   erulesd2s.jar: {'OK' if erulesd2s_jar.exists() else 'FALTANDO (será ignorado)'}")

print("\n" + "="*80)

---
## PARTE 2: CONFIGURAÇÃO DOS EXPERIMENTOS
---

In [ ]:
# CÉLULA 2.1: Configuração dos 3 Experimentos
import numpy as np
import pandas as pd
import json
import subprocess
import time
from datetime import datetime
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURAÇÃO DOS 3 EXPERIMENTOS
# ============================================================

EXPERIMENT_CONFIGS = {
    'exp_a_chunk1000': {
        'chunk_size': 1000,
        'penalty_weight': 0.0,
        'description': 'Baseline configuration (chunk_size=1000)',
        'batches': {
            'batch_1': {
                'base_dir': 'experiments_6chunks_phase2_gbml/batch_1',
                'datasets': [
                    'SEA_Abrupt_Simple', 'SEA_Abrupt_Chain', 'SEA_Abrupt_Recurring',
                    'AGRAWAL_Abrupt_Simple_Mild', 'AGRAWAL_Abrupt_Simple_Severe', 'AGRAWAL_Abrupt_Chain_Long',
                    'RBF_Abrupt_Severe', 'RBF_Abrupt_Blip',
                    'STAGGER_Abrupt_Chain', 'STAGGER_Abrupt_Recurring',
                    'HYPERPLANE_Abrupt_Simple', 'RANDOMTREE_Abrupt_Simple'
                ]
            },
            'batch_2': {
                'base_dir': 'experiments_6chunks_phase2_gbml/batch_2',
                'datasets': [
                    'SEA_Gradual_Simple_Fast', 'SEA_Gradual_Simple_Slow', 'SEA_Gradual_Recurring',
                    'STAGGER_Gradual_Chain',
                    'RBF_Gradual_Moderate', 'RBF_Gradual_Severe',
                    'HYPERPLANE_Gradual_Simple', 'RANDOMTREE_Gradual_Simple', 'LED_Gradual_Simple'
                ]
            },
            'batch_3': {
                'base_dir': 'experiments_6chunks_phase2_gbml/batch_3',
                'datasets': [
                    'SEA_Abrupt_Chain_Noise', 'STAGGER_Abrupt_Chain_Noise',
                    'AGRAWAL_Abrupt_Simple_Severe_Noise', 'SINE_Abrupt_Recurring_Noise',
                    'RBF_Abrupt_Blip_Noise', 'RBF_Gradual_Severe_Noise',
                    'HYPERPLANE_Gradual_Noise', 'RANDOMTREE_Gradual_Noise'
                ]
            },
            'batch_4': {
                'base_dir': 'experiments_6chunks_phase2_gbml/batch_4',
                'datasets': [
                    'SINE_Abrupt_Simple', 'SINE_Gradual_Recurring',
                    'LED_Abrupt_Simple',
                    'WAVEFORM_Abrupt_Simple', 'WAVEFORM_Gradual_Simple',
                    'RANDOMTREE_Abrupt_Recurring'
                ]
            },
            'batch_5': {
                'base_dir': 'experiments_6chunks_phase3_real/batch_5',
                'datasets': ['Electricity', 'Shuttle', 'CovType', 'PokerHand', 'IntelLabSensors']
            },
            'batch_6': {
                'base_dir': 'experiments_6chunks_phase3_real/batch_6',
                'datasets': [
                    'SEA_Stationary', 'AGRAWAL_Stationary', 'RBF_Stationary',
                    'LED_Stationary', 'HYPERPLANE_Stationary', 'RANDOMTREE_Stationary'
                ]
            },
            'batch_7': {
                'base_dir': 'experiments_6chunks_phase3_real/batch_7',
                'datasets': [
                    'STAGGER_Stationary', 'WAVEFORM_Stationary', 'SINE_Stationary',
                    'AssetNegotiation_F2', 'AssetNegotiation_F3', 'AssetNegotiation_F4'
                ]
            }
        }
    },
    'exp_b_chunk2000': {
        'chunk_size': 2000,
        'penalty_weight': 0.0,
        'description': 'Larger evaluation window (chunk_size=2000)',
        'batches': {
            'batch_1': {
                'base_dir': 'experiments_chunk2000_phase1/batch_1',
                'datasets': [
                    'SEA_Abrupt_Simple', 'SEA_Abrupt_Chain', 'SEA_Abrupt_Recurring',
                    'AGRAWAL_Abrupt_Simple_Mild', 'AGRAWAL_Abrupt_Simple_Severe', 'AGRAWAL_Abrupt_Chain_Long',
                    'RBF_Abrupt_Severe', 'RBF_Abrupt_Blip',
                    'STAGGER_Abrupt_Chain', 'STAGGER_Abrupt_Recurring',
                    'HYPERPLANE_Abrupt_Simple', 'RANDOMTREE_Abrupt_Simple'
                ]
            },
            'batch_2': {
                'base_dir': 'experiments_chunk2000_phase1/batch_2',
                'datasets': [
                    'SEA_Gradual_Simple_Fast', 'SEA_Gradual_Simple_Slow', 'SEA_Gradual_Recurring',
                    'STAGGER_Gradual_Chain',
                    'RBF_Gradual_Moderate', 'RBF_Gradual_Severe',
                    'HYPERPLANE_Gradual_Simple', 'RANDOMTREE_Gradual_Simple', 'LED_Gradual_Simple'
                ]
            },
            'batch_3': {
                'base_dir': 'experiments_chunk2000_phase1/batch_3',
                'datasets': [
                    'SEA_Abrupt_Chain_Noise', 'STAGGER_Abrupt_Chain_Noise',
                    'AGRAWAL_Abrupt_Simple_Severe_Noise', 'SINE_Abrupt_Recurring_Noise',
                    'RBF_Abrupt_Blip_Noise', 'RBF_Gradual_Severe_Noise',
                    'HYPERPLANE_Gradual_Noise', 'RANDOMTREE_Gradual_Noise'
                ]
            },
            'batch_4': {
                'base_dir': 'experiments_chunk2000_phase1/batch_4',
                'datasets': [
                    'SINE_Abrupt_Simple', 'SINE_Gradual_Recurring',
                    'LED_Abrupt_Simple',
                    'WAVEFORM_Abrupt_Simple', 'WAVEFORM_Gradual_Simple',
                    'RANDOMTREE_Abrupt_Recurring'
                ]
            },
            'batch_5': {
                'base_dir': 'experiments_chunk2000_phase2/batch_5',
                'datasets': ['Electricity', 'Shuttle', 'CovType', 'PokerHand', 'IntelLabSensors']
            },
            'batch_6': {
                'base_dir': 'experiments_chunk2000_phase2/batch_6',
                'datasets': [
                    'SEA_Stationary', 'AGRAWAL_Stationary', 'RBF_Stationary',
                    'LED_Stationary', 'HYPERPLANE_Stationary', 'RANDOMTREE_Stationary'
                ]
            },
            'batch_7': {
                'base_dir': 'experiments_chunk2000_phase2/batch_7',
                'datasets': [
                    'STAGGER_Stationary', 'WAVEFORM_Stationary', 'SINE_Stationary',
                    'AssetNegotiation_F2', 'AssetNegotiation_F3', 'AssetNegotiation_F4'
                ]
            }
        }
    },
    'exp_c_balanced': {
        'chunk_size': 2000,
        'penalty_weight': 0.1,
        'description': 'Performance-complexity balance (PENALTY_WEIGHT=0.1)',
        'batches': {
            'batch_1': {
                'base_dir': 'experiments_balanced_phase1/batch_1',
                'datasets': [
                    'SEA_Abrupt_Simple', 'SEA_Abrupt_Chain', 'SEA_Abrupt_Recurring',
                    'AGRAWAL_Abrupt_Simple_Mild', 'AGRAWAL_Abrupt_Simple_Severe', 'AGRAWAL_Abrupt_Chain_Long',
                    'RBF_Abrupt_Severe', 'RBF_Abrupt_Blip',
                    'STAGGER_Abrupt_Chain', 'STAGGER_Abrupt_Recurring',
                    'HYPERPLANE_Abrupt_Simple', 'RANDOMTREE_Abrupt_Simple'
                ]
            },
            'batch_2': {
                'base_dir': 'experiments_balanced_phase1/batch_2',
                'datasets': [
                    'SEA_Gradual_Simple_Fast', 'SEA_Gradual_Simple_Slow', 'SEA_Gradual_Recurring',
                    'STAGGER_Gradual_Chain',
                    'RBF_Gradual_Moderate', 'RBF_Gradual_Severe',
                    'HYPERPLANE_Gradual_Simple', 'RANDOMTREE_Gradual_Simple', 'LED_Gradual_Simple'
                ]
            },
            'batch_3': {
                'base_dir': 'experiments_balanced_phase1/batch_3',
                'datasets': [
                    'SEA_Abrupt_Chain_Noise', 'STAGGER_Abrupt_Chain_Noise',
                    'AGRAWAL_Abrupt_Simple_Severe_Noise', 'SINE_Abrupt_Recurring_Noise',
                    'RBF_Abrupt_Blip_Noise', 'RBF_Gradual_Severe_Noise',
                    'HYPERPLANE_Gradual_Noise', 'RANDOMTREE_Gradual_Noise'
                ]
            },
            'batch_4': {
                'base_dir': 'experiments_balanced_phase1/batch_4',
                'datasets': [
                    'SINE_Abrupt_Simple', 'SINE_Gradual_Recurring',
                    'LED_Abrupt_Simple',
                    'WAVEFORM_Abrupt_Simple', 'WAVEFORM_Gradual_Simple',
                    'RANDOMTREE_Abrupt_Recurring'
                ]
            },
            'batch_5': {
                'base_dir': 'experiments_balanced_phase2/batch_5',
                'datasets': ['Electricity', 'Shuttle', 'CovType', 'PokerHand', 'IntelLabSensors']
            },
            'batch_6': {
                'base_dir': 'experiments_balanced_phase2/batch_6',
                'datasets': [
                    'SEA_Stationary', 'AGRAWAL_Stationary', 'RBF_Stationary',
                    'LED_Stationary', 'HYPERPLANE_Stationary', 'RANDOMTREE_Stationary'
                ]
            },
            'batch_7': {
                'base_dir': 'experiments_balanced_phase2/batch_7',
                'datasets': [
                    'STAGGER_Stationary', 'WAVEFORM_Stationary', 'SINE_Stationary',
                    'AssetNegotiation_F2', 'AssetNegotiation_F3', 'AssetNegotiation_F4'
                ]
            }
        }
    }
}

# ============================================================
# SELECIONAR EXPERIMENTOS A EXECUTAR
# ============================================================
# Descomente os experimentos que deseja executar:

EXPERIMENTS_TO_RUN = [
    # 'exp_a_chunk1000',  # EXP-A: Já tem resultados comparativos
    'exp_b_chunk2000',    # EXP-B: FALTA executar comparativos
    'exp_c_balanced',     # EXP-C: FALTA executar comparativos
]

# Modelos a executar
MODELS_TO_RUN = [
    'ROSE_Original',   # ROSE como no paper (métrica global)
    'ROSE_ChunkEval',  # ROSE com avaliação por chunk
    'HAT',             # Hoeffding Adaptive Tree
    'ARF',             # Adaptive Random Forest
    'SRP',             # Streaming Random Patches
    'ACDWM',           # Adaptive Chunk-based DWM (só binário!)
    'ERulesD2S'        # Evolutionary Rules (usa cache)
]

# Timeout por modelo (segundos)
MODEL_TIMEOUT = {
    'ROSE_Original': 600,
    'ROSE_ChunkEval': 600,
    'HAT': 300,
    'ARF': 600,
    'SRP': 600,
    'ACDWM': 600,
    'ERulesD2S': 1800
}

# Controle de cache
USE_CACHE = True  # Se True, usa resultados existentes
FORCE_RERUN = []  # Lista de modelos para forçar re-execução

print("Configuração carregada!")
print(f"Experimentos a executar: {EXPERIMENTS_TO_RUN}")
print(f"Modelos a executar: {MODELS_TO_RUN}")
print(f"Usar cache: {USE_CACHE}")

In [ ]:
# CÉLULA 2.2: Auto-detectar diretórios existentes

def detect_available_datasets(exp_name, config):
    """
    Detecta quais datasets estão disponíveis para um experimento.
    Retorna lista de (batch_name, dataset_name, dataset_dir) para datasets que existem.
    """
    available = []
    
    for batch_name, batch_info in config['batches'].items():
        base_dir = Path(WORK_DIR) / batch_info['base_dir']
        
        if not base_dir.exists():
            continue
            
        for dataset_name in batch_info['datasets']:
            dataset_dir = base_dir / dataset_name
            run_dir = dataset_dir / 'run_1'
            
            # Verificar se tem chunk_data (dados existentes)
            chunk_data_dir = run_dir / 'chunk_data'
            if chunk_data_dir.exists() and any(chunk_data_dir.glob('chunk_*_test.csv')):
                available.append((batch_name, dataset_name, dataset_dir))
    
    return available

# Mostrar resumo
print("="*80)
print("DATASETS DISPONÍVEIS POR EXPERIMENTO")
print("="*80)

for exp_name in EXPERIMENTS_TO_RUN:
    config = EXPERIMENT_CONFIGS[exp_name]
    available = detect_available_datasets(exp_name, config)
    
    print(f"\n{exp_name}: {len(available)} datasets")
    print(f"  chunk_size: {config['chunk_size']}")
    print(f"  Descrição: {config['description']}")
    
    # Agrupar por batch
    by_batch = defaultdict(list)
    for batch, dataset, _ in available:
        by_batch[batch].append(dataset)
    
    for batch, datasets in sorted(by_batch.items()):
        print(f"    {batch}: {len(datasets)} datasets")

---
## PARTE 3: FUNÇÕES AUXILIARES
---

In [ ]:
# CÉLULA 3.1: Funções para carregar dados

def load_chunks_from_gbml(dataset_dir):
    """
    Carrega chunks de dados do diretório GBML.
    Formato: chunk_data/chunk_X_test.csv (X = 1, 2, 3, ...)
    """
    dataset_dir = Path(dataset_dir)
    run_dir = dataset_dir / "run_1"
    chunk_data_dir = run_dir / "chunk_data"

    X_chunks = []
    y_chunks = []

    if not chunk_data_dir.exists():
        return None, None

    # Procurar chunks de teste
    test_files = sorted(chunk_data_dir.glob("chunk_*_test.csv"))

    for test_file in test_files:
        try:
            df = pd.read_csv(test_file)

            # Identificar coluna de classe
            if 'class' in df.columns:
                y = df['class'].values
                X = df.drop(columns=['class']).values
            elif 'label' in df.columns:
                y = df['label'].values
                X = df.drop(columns=['label']).values
            else:
                y = df.iloc[:, -1].values
                X = df.iloc[:, :-1].values

            X_chunks.append(X.astype(float))
            y_chunks.append(y.astype(int))
        except Exception as e:
            print(f"  Erro ao carregar {test_file}: {e}")
            continue

    if len(X_chunks) > 0:
        return X_chunks, y_chunks
    return None, None


def load_existing_model_results(dataset_dir, model_name):
    """
    Carrega resultados existentes de um modelo comparativo.
    Retorna dict com gmean ou None.
    """
    dataset_dir = Path(dataset_dir)
    run_dir = dataset_dir / "run_1"

    # Mapear nome do modelo para arquivo
    file_map = {
        'HAT': 'river_HAT_results.csv',
        'ARF': 'river_ARF_results.csv',
        'SRP': 'river_SRP_results.csv',
        'ACDWM': 'acdwm_results.csv',
        'ERulesD2S': 'erulesd2s_results.csv',
        'ROSE_Original': 'rose_original_results.csv',
        'ROSE_ChunkEval': 'rose_chunk_eval_results.csv'
    }

    if model_name not in file_map:
        return None

    result_file = run_dir / file_map[model_name]

    if result_file.exists():
        try:
            df = pd.read_csv(result_file)

            # Procurar coluna de gmean
            for col in ['test_gmean', 'gmean', 'G-Mean', 'g_mean']:
                if col in df.columns:
                    return {'gmean': df[col].mean(), 'source': 'cached'}

            # Fallback: accuracy
            if 'accuracy' in df.columns or 'test_accuracy' in df.columns:
                col = 'test_accuracy' if 'test_accuracy' in df.columns else 'accuracy'
                return {'gmean': df[col].mean(), 'source': 'cached_accuracy'}

        except Exception as e:
            pass

    return None


print("Funções de carregamento definidas!")

In [ ]:
# CÉLULA 3.2: Funções para ARFF e ROSE

def get_rose_evaluator(n_classes):
    """Retorna o evaluator correto baseado no número de classes."""
    if n_classes <= 2:
        return "WindowAUCImbalancedPerformanceEvaluator"
    else:
        return "WindowAUCMultiClassImbalancedPerformanceEvaluator"


def create_arff_file(X, y, output_path, relation_name="stream"):
    """Cria arquivo ARFF a partir de dados numpy."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    n_samples, n_features = X.shape
    classes = sorted(np.unique(y))

    with open(output_path, 'w') as f:
        f.write(f"@relation {relation_name}\n\n")
        for i in range(n_features):
            f.write(f"@attribute attr{i} numeric\n")
        class_str = ",".join([str(c) for c in classes])
        f.write(f"@attribute class {{{class_str}}}\n\n")
        f.write("@data\n")
        for i in range(n_samples):
            row = ",".join([str(x) for x in X[i]]) + f",{y[i]}\n"
            f.write(row)

    return output_path


def run_rose_original(arff_file, output_dir, n_classes=2, timeout=600):
    """ROSE_Original: Executa ROSE como no paper original (métrica global)."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_file = output_dir / "rose_original_results.csv"
    log_file = output_dir / "rose_original_log.txt"

    classpath = "rose_jars/ROSE-1.0.jar:rose_jars/MOA-dependencies.jar"
    evaluator = get_rose_evaluator(n_classes)

    cmd = [
        "java", "-Xmx4g",
        f"-javaagent:rose_jars/sizeofag-1.0.4.jar",
        "-cp", classpath, "moa.DoTask"
    ]

    task_parts = [
        "EvaluateInterleavedTestThenTrain",
        "-e", f"({evaluator})",
        "-s", f"(ArffFileStream -f {arff_file})",
        "-l", "(moa.classifiers.meta.imbalanced.ROSE)",
        "-f", "500",
        "-d", str(output_file)
    ]

    task_string = " ".join(task_parts)
    cmd.append(task_string)

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)

        with open(log_file, 'w') as f:
            f.write(f"Command: {' '.join(cmd)}\n\n")
            f.write(f"STDOUT:\n{result.stdout}\n\nSTDERR:\n{result.stderr}")

        if result.returncode != 0:
            return False, {'gmean': 0.0, 'error': 'Non-zero return code'}

        results = parse_rose_results_global(output_file)
        return True, results

    except subprocess.TimeoutExpired:
        return False, {'gmean': 0.0, 'error': 'Timeout'}
    except Exception as e:
        return False, {'gmean': 0.0, 'error': str(e)}


def run_rose_chunk_eval(arff_file, output_dir, n_classes=2, chunk_size=1000, timeout=600):
    """ROSE_ChunkEval: Executa ROSE com avaliação por chunk (média)."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_file = output_dir / "rose_chunk_eval_results.csv"
    log_file = output_dir / "rose_chunk_eval_log.txt"

    classpath = "rose_jars/ROSE-1.0.jar:rose_jars/MOA-dependencies.jar"
    evaluator = get_rose_evaluator(n_classes)

    cmd = [
        "java", "-Xmx4g",
        f"-javaagent:rose_jars/sizeofag-1.0.4.jar",
        "-cp", classpath, "moa.DoTask"
    ]

    task_parts = [
        "EvaluateInterleavedTestThenTrain",
        "-e", f"({evaluator})",
        "-s", f"(ArffFileStream -f {arff_file})",
        "-l", "(moa.classifiers.meta.imbalanced.ROSE)",
        "-f", str(chunk_size),
        "-d", str(output_file)
    ]

    task_string = " ".join(task_parts)
    cmd.append(task_string)

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)

        with open(log_file, 'w') as f:
            f.write(f"Command: {' '.join(cmd)}\n\n")
            f.write(f"STDOUT:\n{result.stdout}\n\nSTDERR:\n{result.stderr}")

        if result.returncode != 0:
            return False, {'gmean': 0.0, 'error': 'Non-zero return code'}

        results = parse_rose_results_chunk_avg(output_file)
        return True, results

    except subprocess.TimeoutExpired:
        return False, {'gmean': 0.0, 'error': 'Timeout'}
    except Exception as e:
        return False, {'gmean': 0.0, 'error': str(e)}


def parse_rose_results_global(results_file):
    """Parseia resultados do ROSE - retorna métrica GLOBAL (última linha)."""
    results = {'gmean': 0.0, 'accuracy': 0.0}
    results_file = Path(results_file)

    if not results_file.exists():
        return results

    try:
        with open(results_file, 'r') as f:
            lines = f.readlines()

        data_lines = [line for line in lines
                      if not line.startswith('Learner')
                      and not line.startswith('learning')
                      and line.strip()]

        if len(data_lines) > 0:
            header_line = lines[0]
            last_data_line = data_lines[-1]

            headers = header_line.strip().split(',')
            values = last_data_line.strip().split(',')
            data_dict = dict(zip(headers, values))

            if 'G-Mean' in data_dict:
                try:
                    results['gmean'] = float(data_dict['G-Mean'])
                except:
                    pass
            if 'Accuracy' in data_dict:
                try:
                    results['accuracy'] = float(data_dict['Accuracy'])
                except:
                    pass

    except Exception as e:
        results['error'] = str(e)

    return results


def parse_rose_results_chunk_avg(results_file):
    """Parseia resultados do ROSE - retorna MÉDIA das métricas por chunk."""
    results = {'gmean': 0.0, 'accuracy': 0.0}
    results_file = Path(results_file)

    if not results_file.exists():
        return results

    try:
        with open(results_file, 'r') as f:
            lines = f.readlines()

        data_lines = [line for line in lines
                      if not line.startswith('Learner')
                      and not line.startswith('learning')
                      and line.strip()]

        if len(data_lines) > 0:
            header_line = lines[0]
            headers = header_line.strip().split(',')

            gmeans = []
            accuracies = []

            for data_line in data_lines:
                values = data_line.strip().split(',')
                data_dict = dict(zip(headers, values))

                if 'G-Mean' in data_dict:
                    try:
                        gmeans.append(float(data_dict['G-Mean']))
                    except:
                        pass
                if 'Accuracy' in data_dict:
                    try:
                        accuracies.append(float(data_dict['Accuracy']))
                    except:
                        pass

            if gmeans:
                results['gmean'] = np.mean(gmeans)
            if accuracies:
                results['accuracy'] = np.mean(accuracies)
            results['n_chunks_evaluated'] = len(gmeans)

    except Exception as e:
        results['error'] = str(e)

    return results


print("Funções ROSE definidas!")

In [ ]:
# CÉLULA 3.3: Funções para River Models (HAT, ARF, SRP)

from river import tree, ensemble, forest
from sklearn.metrics import accuracy_score, f1_score

def calculate_gmean(y_true, y_pred):
    """Calcula G-mean."""
    from sklearn.metrics import confusion_matrix
    classes = np.unique(y_true)

    if len(classes) == 2:
        cm = confusion_matrix(y_true, y_pred, labels=classes)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
            return np.sqrt(sensitivity * specificity)

    # Multi-class
    recalls = []
    for c in classes:
        mask = y_true == c
        if mask.sum() > 0:
            recall = np.mean(y_pred[mask] == c)
            recalls.append(recall)

    if len(recalls) > 0:
        return np.prod(recalls) ** (1.0 / len(recalls))
    return 0.0


def run_river_model(model_name, X_chunks, y_chunks, timeout=300):
    """
    Executa modelo River (HAT, ARF, SRP) nos chunks.
    Retorna métricas por chunk e global.
    """
    try:
        # Criar modelo
        if model_name == 'HAT':
            model = tree.HoeffdingAdaptiveTreeClassifier()
        elif model_name == 'ARF':
            # NOTA: Em River >= 0.21, ARF foi movido de ensemble para forest
            model = forest.ARFClassifier(n_models=10)
        elif model_name == 'SRP':
            model = ensemble.SRPClassifier(n_models=10)
        else:
            return {'gmean': 0.0, 'error': f'Unknown model: {model_name}'}

        all_preds = []
        all_true = []
        chunk_results = []

        for chunk_idx, (X, y) in enumerate(zip(X_chunks, y_chunks)):
            chunk_preds = []

            for i in range(len(X)):
                x_dict = {f'f{j}': float(X[i, j]) for j in range(X.shape[1])}
                y_i = int(y[i])

                # Predict
                pred = model.predict_one(x_dict)
                if pred is None:
                    pred = 0
                chunk_preds.append(pred)

                # Learn
                model.learn_one(x_dict, y_i)

            all_preds.extend(chunk_preds)
            all_true.extend(y)

            # Chunk metrics
            chunk_gmean = calculate_gmean(np.array(y), np.array(chunk_preds))
            chunk_acc = accuracy_score(y, chunk_preds)
            chunk_results.append({
                'chunk': chunk_idx + 1,
                'test_gmean': chunk_gmean,
                'test_accuracy': chunk_acc
            })

        # Global metrics
        all_preds = np.array(all_preds)
        all_true = np.array(all_true)

        gmean = calculate_gmean(all_true, all_preds)
        accuracy = accuracy_score(all_true, all_preds)

        return {
            'gmean': gmean,
            'accuracy': accuracy,
            'chunk_results': chunk_results
        }

    except Exception as e:
        return {'gmean': 0.0, 'accuracy': 0.0, 'error': str(e)}


print("Funções River definidas!")

In [ ]:
# CÉLULA 3.4: Funções para ACDWM

def run_acdwm(X_chunks, y_chunks, acdwm_path="/content/ACDWM", timeout=600):
    """
    Executa ACDWM nos chunks.
    LIMITAÇÃO: ACDWM só funciona com problemas BINÁRIOS (2 classes).
    """
    try:
        import sys
        if acdwm_path not in sys.path:
            sys.path.insert(0, acdwm_path)

        # Verificar número de classes
        all_y = np.concatenate(y_chunks)
        unique_classes = np.unique(all_y)
        n_classes = len(unique_classes)

        if n_classes > 2:
            return {
                'gmean': 0.0,
                'accuracy': 0.0,
                'error': f'ACDWM does not support multiclass (n_classes={n_classes})'
            }

        if n_classes < 2:
            return {
                'gmean': 0.0,
                'accuracy': 0.0,
                'error': f'Need at least 2 classes (found {n_classes})'
            }

        from dwmil import DWMIL

        model = DWMIL(
            data_num=999999,
            chunk_size=0,
            theta=0.001,
            err_func='gm',
            r=1.0
        )

        all_preds = []
        all_true = []
        chunk_results = []

        def to_acdwm_labels(y):
            return np.where(y == 0, -1, 1).astype(np.int32)

        def from_acdwm_labels(y):
            return np.where(y == -1, 0, 1).astype(np.int32)

        for chunk_idx, (X, y) in enumerate(zip(X_chunks, y_chunks)):
            X = np.array(X, dtype=float)
            y = np.array(y, dtype=int)
            y_acdwm = to_acdwm_labels(y)

            try:
                y_pred_acdwm = model.update_chunk(X, y_acdwm)
                y_pred = from_acdwm_labels(y_pred_acdwm)

                all_preds.extend(y_pred)
                all_true.extend(y)

                # Chunk metrics
                chunk_gmean = calculate_gmean(y, y_pred)
                chunk_acc = accuracy_score(y, y_pred)
                chunk_results.append({
                    'chunk': chunk_idx + 1,
                    'test_gmean': chunk_gmean,
                    'test_accuracy': chunk_acc
                })

            except Exception as e:
                return {
                    'gmean': 0.0,
                    'accuracy': 0.0,
                    'error': f'ACDWM failed on chunk {chunk_idx}: {str(e)[:50]}'
                }

        if len(all_preds) == 0:
            return {'gmean': 0.0, 'accuracy': 0.0, 'error': 'No predictions made'}

        all_preds = np.array(all_preds)
        all_true = np.array(all_true)

        gmean = calculate_gmean(all_true, all_preds)
        accuracy = accuracy_score(all_true, all_preds)

        return {
            'gmean': gmean,
            'accuracy': accuracy,
            'chunk_results': chunk_results
        }

    except ImportError as e:
        return {
            'gmean': 0.0,
            'accuracy': 0.0,
            'error': f'Could not import DWMIL: {str(e)[:50]}'
        }
    except Exception as e:
        return {
            'gmean': 0.0,
            'accuracy': 0.0,
            'error': f'ACDWM error: {str(e)[:50]}'
        }


print("Funções ACDWM definidas!")

In [ ]:
# CÉLULA 3.5: Funções para salvar resultados

def save_model_results(dataset_dir, model_name, results):
    """
    Salva resultados de um modelo no diretório do dataset.
    Formato: run_1/river_<model>_results.csv ou run_1/<model>_results.csv
    """
    dataset_dir = Path(dataset_dir)
    run_dir = dataset_dir / "run_1"
    run_dir.mkdir(parents=True, exist_ok=True)

    # Nome do arquivo de saída
    if model_name in ['HAT', 'ARF', 'SRP']:
        output_file = run_dir / f"river_{model_name}_results.csv"
    elif model_name == 'ACDWM':
        output_file = run_dir / "acdwm_results.csv"
    elif model_name == 'ROSE_Original':
        output_file = run_dir / "rose_original_results.csv"
    elif model_name == 'ROSE_ChunkEval':
        output_file = run_dir / "rose_chunk_eval_results.csv"
    else:
        output_file = run_dir / f"{model_name.lower()}_results.csv"

    # Se temos resultados por chunk, salvar detalhado
    if 'chunk_results' in results and results['chunk_results']:
        df = pd.DataFrame(results['chunk_results'])
        df.to_csv(output_file, index=False)
    else:
        # Salvar resultado global
        df = pd.DataFrame([{
            'chunk': 1,
            'test_gmean': results.get('gmean', 0.0),
            'test_accuracy': results.get('accuracy', 0.0)
        }])
        df.to_csv(output_file, index=False)

    return output_file


print("Funções de salvamento definidas!")

In [ ]:
# CÉLULA 3.6: Funções para ERulesD2S (EXECUÇÃO REAL)

# Flag para controlar execução do ERulesD2S
ERULESD2S_ENABLED = True  # Mude para False se quiser apenas usar cache
ERULESD2S_JAR = Path(WORK_DIR) / "erulesd2s.jar"
ERULESD2S_JCLEC_JAR = Path(WORK_DIR) / "lib" / "JCLEC4-base-1.0-jar-with-dependencies.jar"

def run_erulesd2s(X_chunks, y_chunks, dataset_dir, dataset_name, chunk_size=1000, timeout=600):
    """
    Executa ERulesD2S nos chunks usando o wrapper Java/MOA.

    IMPORTANTE: Baseado no erulesd2s_wrapper.py que funciona corretamente.

    Args:
        X_chunks: Lista de arrays X (features)
        y_chunks: Lista de arrays y (labels)
        dataset_dir: Diretório do dataset
        dataset_name: Nome do dataset
        chunk_size: Tamanho do chunk para avaliação
        timeout: Timeout em segundos (padrão: 10 min - ajustado para velocidade)

    Returns:
        dict com 'gmean', 'accuracy', 'chunk_results' ou 'error'
    """
    import shlex

    # Verificar se JAR existe
    if not ERULESD2S_JAR.exists():
        return {'gmean': 0.0, 'error': f'erulesd2s.jar not found at {ERULESD2S_JAR}'}

    dataset_dir = Path(dataset_dir)
    run_dir = dataset_dir / "run_1"
    run_dir.mkdir(parents=True, exist_ok=True)

    # Concatenar todos os chunks em um único dataset
    X_all = np.vstack(X_chunks)
    y_all = np.concatenate(y_chunks)

    # Criar arquivo ARFF
    arff_dir = run_dir / "erulesd2s_arff"
    arff_dir.mkdir(parents=True, exist_ok=True)
    arff_file = arff_dir / f"{dataset_name}.arff"
    create_arff_file(X_all, y_all, arff_file, relation_name=dataset_name)

    # Diretório de saída
    output_dir = run_dir / "erulesd2s_run"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / "erulesd2s_output.csv"
    log_file = output_dir / "erulesd2s_log.txt"

    # Construir classpath (Linux usa :)
    classpath_parts = [str(ERULESD2S_JAR)]
    if ERULESD2S_JCLEC_JAR.exists():
        classpath_parts.append(str(ERULESD2S_JCLEC_JAR))

    # Adicionar outras JARs na pasta lib/
    lib_dir = Path(WORK_DIR) / "lib"
    if lib_dir.exists():
        for jar in lib_dir.glob("*.jar"):
            if str(jar) not in classpath_parts:
                classpath_parts.append(str(jar))

    classpath = ":".join(classpath_parts)

    # Parâmetros ERulesD2S (REDUZIDOS para velocidade)
    population_size = 15   # Reduzido de 25
    num_generations = 25   # Reduzido de 50
    rules_per_class = 3    # Reduzido de 5

    # Construir comando como lista (formato correto para subprocess)
    # IMPORTANTE: A task string precisa ser um único argumento para moa.DoTask
    learner = f"(moa.classifiers.evolutionary.EvolutionaryRuleLearner -s {population_size} -g {num_generations} -r {rules_per_class})"
    stream = f"(ArffFileStream -f {arff_file})"

    task_string = f"EvaluateInterleavedTestThenTrain -s {stream} -l {learner} -f {chunk_size} -d {output_file}"

    cmd = [
        "java",
        "-Xmx4g",
        "-cp", classpath,
        "moa.DoTask",
        task_string
    ]

    try:
        print(f"    Executando ERulesD2S (timeout={timeout}s)...")
        start_time = time.time()

        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=timeout,
            cwd=str(Path(WORK_DIR))
        )

        duration = time.time() - start_time

        # Salvar log
        with open(log_file, 'w') as f:
            f.write(f"Command: {' '.join(cmd)}\n\n")
            f.write(f"Duration: {duration:.1f}s\n\n")
            f.write(f"Return code: {result.returncode}\n\n")
            f.write(f"STDOUT:\n{result.stdout}\n\n")
            f.write(f"STDERR:\n{result.stderr}\n")

        if result.returncode != 0:
            error_msg = result.stderr[:200] if result.stderr else "Unknown error"
            return {
                'gmean': 0.0,
                'error': f'returncode={result.returncode}: {error_msg}'
            }

        # Parsear resultados
        if output_file.exists():
            try:
                with open(output_file, 'r') as f:
                    lines = f.readlines()

                # Filtrar linhas de dados (não headers)
                data_lines = [line for line in lines
                              if not line.startswith('Learner')
                              and not line.startswith('learning')
                              and line.strip()]

                if data_lines:
                    header_line = lines[0]
                    headers = header_line.strip().split(',')

                    accuracies = []
                    chunk_results = []

                    for chunk_idx, data_line in enumerate(data_lines):
                        values = data_line.strip().split(',')
                        data_dict = dict(zip(headers, values))

                        # Extrair accuracy
                        acc = 0.0
                        if 'classifications correct (percent)' in data_dict:
                            try:
                                acc = float(data_dict['classifications correct (percent)']) / 100.0
                            except:
                                pass

                        accuracies.append(acc)
                        chunk_results.append({
                            'chunk': chunk_idx + 1,
                            'test_gmean': acc,
                            'test_accuracy': acc
                        })

                    # Calcular média
                    gmean = np.mean(accuracies) if accuracies else 0.0

                    # Salvar resultados no formato esperado
                    results_file = run_dir / "erulesd2s_results.csv"
                    df = pd.DataFrame(chunk_results)
                    df['model'] = 'ERulesD2S'
                    df['execution_time'] = duration / len(chunk_results) if chunk_results else 0
                    df.to_csv(results_file, index=False)

                    print(f"    ERulesD2S concluído em {duration:.1f}s (gmean={gmean:.4f})")

                    return {
                        'gmean': gmean,
                        'accuracy': gmean,
                        'chunk_results': chunk_results,
                        'execution_time': duration
                    }

            except Exception as e:
                return {'gmean': 0.0, 'error': f'Error parsing: {str(e)[:50]}'}

        return {'gmean': 0.0, 'error': 'No output file'}

    except subprocess.TimeoutExpired:
        print(f"    ERulesD2S TIMEOUT após {timeout}s")
        return {'gmean': 0.0, 'error': f'Timeout ({timeout}s)'}
    except Exception as e:
        return {'gmean': 0.0, 'error': f'Exception: {str(e)[:50]}'}


# Verificar se ERulesD2S está disponível
print("Verificação ERulesD2S:")
print(f"  erulesd2s.jar: {'OK' if ERULESD2S_JAR.exists() else 'FALTANDO'}")
print(f"  JCLEC4 JAR: {'OK' if ERULESD2S_JCLEC_JAR.exists() else 'FALTANDO'}")
print(f"  ERULESD2S_ENABLED: {ERULESD2S_ENABLED}")

if not ERULESD2S_JAR.exists():
    print("\n  AVISO: ERulesD2S não será executado (JAR não encontrado)")
    print("  Apenas resultados em cache serão usados")
    ERULESD2S_ENABLED = False

print("\nFunções ERulesD2S definidas!")

---
## PARTE 4: EXECUÇÃO DOS EXPERIMENTOS
---

In [ ]:
# CÉLULA 4.1: Executar TODOS os Modelos em TODOS os Experimentos

ALL_RESULTS = []

print("="*80)
print("INÍCIO DA EXECUÇÃO - TODOS OS MODELOS EM TODOS OS EXPERIMENTOS")
print("="*80)
print(f"Início: {datetime.now()}")
print(f"Experimentos: {EXPERIMENTS_TO_RUN}")
print(f"Modelos: {MODELS_TO_RUN}")
print(f"Usar cache: {USE_CACHE}")
print("="*80)

total_start = time.time()

for exp_name in EXPERIMENTS_TO_RUN:
    config = EXPERIMENT_CONFIGS[exp_name]
    chunk_size = config['chunk_size']
    
    print(f"\n{'#'*80}")
    print(f"# EXPERIMENTO: {exp_name}")
    print(f"# chunk_size: {chunk_size}")
    print(f"# {config['description']}")
    print(f"{'#'*80}")
    
    # Detectar datasets disponíveis
    available = detect_available_datasets(exp_name, config)
    print(f"\nDatasets disponíveis: {len(available)}")
    
    for idx, (batch_name, dataset_name, dataset_dir) in enumerate(available):
        print(f"\n[{idx+1}/{len(available)}] {batch_name}/{dataset_name}")
        
        # Carregar chunks
        X_chunks, y_chunks = load_chunks_from_gbml(dataset_dir)
        
        if X_chunks is None:
            print(f"  AVISO: Chunks não encontrados")
            continue
        
        n_classes = len(np.unique(np.concatenate(y_chunks)))
        print(f"  Chunks: {len(X_chunks)} | Samples/chunk: ~{len(X_chunks[0])} | Classes: {n_classes}")
        
        # Executar cada modelo
        for model_name in MODELS_TO_RUN:
            # Verificar cache
            if USE_CACHE and model_name not in FORCE_RERUN:
                cached = load_existing_model_results(dataset_dir, model_name)
                if cached:
                    ALL_RESULTS.append({
                        'experiment': exp_name,
                        'batch': batch_name,
                        'dataset': dataset_name,
                        'model': model_name,
                        'gmean': cached['gmean'],
                        'status': 'CACHED'
                    })
                    print(f"  {model_name}: {cached['gmean']:.4f} (cached)")
                    continue
            
            # Executar modelo
            try:
                if model_name == 'ROSE_Original':
                    X_all = np.vstack(X_chunks)
                    y_all = np.concatenate(y_chunks)
                    arff_dir = dataset_dir / "run_1" / "rose_arff"
                    arff_file = arff_dir / f"{dataset_name}.arff"
                    create_arff_file(X_all, y_all, arff_file, relation_name=dataset_name)
                    
                    rose_output = dataset_dir / "run_1" / "rose_original_output"
                    success, results = run_rose_original(
                        arff_file, rose_output, n_classes=n_classes,
                        timeout=MODEL_TIMEOUT['ROSE_Original']
                    )
                    gmean = results.get('gmean', 0.0) if success else 0.0
                    status = 'OK' if success else 'FAILED'
                    
                    # Copiar resultado para run_1/ (onde unified_analysis.py espera)
                    if success:
                        import shutil
                        src = rose_output / "rose_original_results.csv"
                        dst = dataset_dir / "run_1" / "rose_original_results.csv"
                        if src.exists():
                            shutil.copy(src, dst)
                    
                elif model_name == 'ROSE_ChunkEval':
                    X_all = np.vstack(X_chunks)
                    y_all = np.concatenate(y_chunks)
                    arff_dir = dataset_dir / "run_1" / "rose_arff"
                    arff_file = arff_dir / f"{dataset_name}.arff"
                    if not arff_file.exists():
                        create_arff_file(X_all, y_all, arff_file, relation_name=dataset_name)
                    
                    rose_output = dataset_dir / "run_1" / "rose_chunk_eval_output"
                    success, results = run_rose_chunk_eval(
                        arff_file, rose_output, n_classes=n_classes,
                        chunk_size=chunk_size,
                        timeout=MODEL_TIMEOUT['ROSE_ChunkEval']
                    )
                    gmean = results.get('gmean', 0.0) if success else 0.0
                    status = 'OK' if success else 'FAILED'
                    
                    # Copiar resultado para run_1/ (onde unified_analysis.py espera)
                    if success:
                        import shutil
                        src = rose_output / "rose_chunk_eval_results.csv"
                        dst = dataset_dir / "run_1" / "rose_chunk_eval_results.csv"
                        if src.exists():
                            shutil.copy(src, dst)
                    
                elif model_name in ['HAT', 'ARF', 'SRP']:
                    results = run_river_model(
                        model_name, X_chunks, y_chunks,
                        timeout=MODEL_TIMEOUT[model_name]
                    )
                    gmean = results.get('gmean', 0.0)
                    status = 'OK' if 'error' not in results else 'FAILED'
                    
                    # Salvar resultados por chunk
                    if 'chunk_results' in results:
                        save_model_results(dataset_dir, model_name, results)
                    
                elif model_name == 'ACDWM':
                    results = run_acdwm(
                        X_chunks, y_chunks,
                        acdwm_path=ACDWM_DIR,
                        timeout=MODEL_TIMEOUT['ACDWM']
                    )
                    gmean = results.get('gmean', 0.0)
                    status = 'OK' if 'error' not in results else 'FAILED'
                    
                    if 'chunk_results' in results:
                        save_model_results(dataset_dir, model_name, results)
                    
                elif model_name == 'ERulesD2S':
                    # ERulesD2S - tenta cache primeiro, depois executa se habilitado
                    cached = load_existing_model_results(dataset_dir, 'ERulesD2S')
                    if cached:
                        gmean = cached['gmean']
                        status = 'CACHED'
                    elif ERULESD2S_ENABLED:
                        # Executar ERulesD2S
                        results = run_erulesd2s(
                            X_chunks, y_chunks, dataset_dir, dataset_name,
                            chunk_size=chunk_size,
                            timeout=MODEL_TIMEOUT['ERulesD2S']
                        )
                        gmean = results.get('gmean', 0.0)
                        status = 'OK' if 'error' not in results else f"FAILED: {results.get('error', '')[:20]}"
                    else:
                        gmean = 0.0
                        status = 'SKIPPED (ERulesD2S disabled)'
                else:
                    gmean = 0.0
                    status = 'UNKNOWN_MODEL'
                
                ALL_RESULTS.append({
                    'experiment': exp_name,
                    'batch': batch_name,
                    'dataset': dataset_name,
                    'model': model_name,
                    'gmean': gmean,
                    'status': status
                })
                print(f"  {model_name}: {gmean:.4f} ({status})")
                
            except Exception as e:
                ALL_RESULTS.append({
                    'experiment': exp_name,
                    'batch': batch_name,
                    'dataset': dataset_name,
                    'model': model_name,
                    'gmean': 0.0,
                    'status': f'ERROR: {str(e)[:30]}'
                })
                print(f"  {model_name}: 0.0000 (ERROR)")

total_time = time.time() - total_start
print(f"\n{'='*80}")
print(f"EXECUÇÃO COMPLETA!")
print(f"Tempo total: {total_time/60:.1f} minutos")
print(f"Total de resultados: {len(ALL_RESULTS)}")
print(f"{'='*80}")

In [ ]:
# CÉLULA 4.2: Salvar Resultados Consolidados

# Criar DataFrame
df_results = pd.DataFrame(ALL_RESULTS)

# Salvar por experimento
for exp_name in df_results['experiment'].unique():
    exp_df = df_results[df_results['experiment'] == exp_name]
    output_file = Path(WORK_DIR) / f"comparative_results_{exp_name}.csv"
    exp_df.to_csv(output_file, index=False)
    print(f"Salvo: {output_file} ({len(exp_df)} registros)")

# Salvar consolidado
all_output = Path(WORK_DIR) / "comparative_results_all_experiments.csv"
df_results.to_csv(all_output, index=False)
print(f"\nConsolidado: {all_output} ({len(df_results)} registros)")

# Resumo
print("\n" + "="*80)
print("RESUMO POR EXPERIMENTO E MODELO")
print("="*80)

summary = df_results.groupby(['experiment', 'model'])['gmean'].agg(['mean', 'std', 'count'])
print(summary.round(4))

---
## PARTE 5: SINCRONIZAR COM DRIVE
---

In [ ]:
# CÉLULA 5.1: Copiar resultados de volta para o Drive

print("Salvando resultados no Drive...")

# Copiar CSVs consolidados
!cp {WORK_DIR}/comparative_results_*.csv "{DRIVE_PATH}/"

# Sincronizar diretórios de experimentos (preservando estrutura)
for exp_name in EXPERIMENTS_TO_RUN:
    config = EXPERIMENT_CONFIGS[exp_name]
    for batch_name, batch_info in config['batches'].items():
        src_dir = Path(WORK_DIR) / batch_info['base_dir']
        dst_dir = Path(DRIVE_PATH) / batch_info['base_dir']
        
        if src_dir.exists():
            print(f"Sincronizando: {batch_info['base_dir']}")
            !rsync -av --update "{src_dir}/" "{dst_dir}/" 2>/dev/null || cp -r "{src_dir}/" "{dst_dir}/"

print("\nSincronização completa!")
print(f"Resultados salvos em: {DRIVE_PATH}")

In [ ]:
# CÉLULA 5.2: Verificar arquivos no Drive

print("Arquivos de resultados no Drive:")
!ls -la "{DRIVE_PATH}"/*.csv 2>/dev/null | head -20

print("\nPronto para executar unified_analysis.py para atualizar tabelas e figuras!")